# Phase 6 — real-capture demo: Mip-NeRF 360 "garden"

Qualitative demo of the full pipeline on a real capture (no clean plates, so no
metrics): fit 3DGS on the garden scene, render an 81-view orbit, remove the
table vase with its shadow and tabletop reflection via SAM 2 clicks + ROSE,
refit, and render a before/after side-by-side.

Session budget: ~15 min download, ~1 h splatfacto (A100), ~20 min ROSE,
~15 min refit. Run top to bottom; the click cell is the only manual step.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/light-footprint-removal'

In [ ]:
# Install. Same pins as phases 2-5. If Colab asks to restart: restart,
# re-run cell 1, skip this cell.
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
!pip -q install nerfstudio
!pip -q install "diffusers>=0.34" accelerate ftfy imageio imageio-ffmpeg
!pip -q install --force-reinstall "numpy==2.0.2"

In [ ]:
# Mip-NeRF 360 'garden': vase on a glossy table, capture path is itself an
# orbit. The official zip holds all scenes (~12 GB); we extract garden only.
import os
if not os.path.isdir('/content/data/garden'):
    os.makedirs('/content/data', exist_ok=True)
    !wget -q --show-progress http://storage.googleapis.com/gresearch/refraw360/360_v2.zip -O /content/360_v2.zip
    !cd /content/data && unzip -q /content/360_v2.zip 'garden/*'
    !rm /content/360_v2.zip
!ls /content/data/garden

In [ ]:
# Fit splatfacto on the real capture (COLMAP poses ship with the dataset).
# Real scene, so unlike the synthetic phases: default 30k iters, SfM point init.
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-train splatfacto \
  --data /content/data/garden --output-dir /content/out_garden \
  --viewer.quit-on-train-completion True --vis tensorboard \
  colmap --downscale-factor 4

In [ ]:
# Render an 81-view orbit at the editors' native 832x480.
# Cameras: 81 evenly spaced training views in capture order. Intrinsics are
# rescaled per axis (mild anamorphic stretch); the refit uses the SAME cameras,
# so the demo stays self-consistent.
import glob, json, math, os
import numpy as np, torch
from pathlib import Path
from PIL import Image
from nerfstudio.utils.eval_utils import eval_setup
from nerfstudio.cameras.cameras import Cameras, CameraType

CONFIG = sorted(glob.glob('/content/out_garden/**/config.yml', recursive=True),
                key=os.path.getmtime)[-1]
_, pipeline, _, _ = eval_setup(Path(CONFIG), test_mode='inference')
ds = pipeline.datamanager.train_dataset
order = np.argsort([str(p) for p in ds.image_filenames])
sel = order[np.linspace(0, len(order) - 1, 81).round().astype(int)]

cams = ds.cameras
W, H = 832, 480
fx = float(cams.fx[sel[0]]) * W / float(cams.width[sel[0]])
fy = float(cams.fy[sel[0]]) * H / float(cams.height[sel[0]])
os.makedirs('/content/orbit_frames', exist_ok=True)
frames = []
for i, ci in enumerate(sel, start=1):
    c2w = cams.camera_to_worlds[ci]
    cam = Cameras(camera_to_worlds=c2w[None], fx=fx, fy=fy, cx=W / 2, cy=H / 2,
                  width=W, height=H,
                  camera_type=CameraType.PERSPECTIVE).to(pipeline.device)
    with torch.no_grad():
        rgb = pipeline.model.get_outputs_for_camera(cam)['rgb'].cpu().numpy()
    Image.fromarray((rgb * 255).clip(0, 255).astype('uint8')).save(
        f'/content/orbit_frames/{i:04d}.jpg')
    m = np.eye(4); m[:3, :4] = c2w.numpy()
    frames.append({'file_path': f'rgb/{i:04d}', 'transform_matrix': m.tolist()})

json.dump({'camera_angle_x': 2 * math.atan(W / (2 * fx)), 'frames': frames},
          open('/content/orbit_transforms.json', 'w'))

from diffusers.utils import export_to_video
video = [Image.open(p).convert('RGB')
         for p in sorted(glob.glob('/content/orbit_frames/*.jpg'))]
export_to_video(video, '/content/orbit.mp4', fps=16)

del pipeline; torch.cuda.empty_cache()   # free VRAM for SAM2 / ROSE
print('rendered 81 orbit frames')

In [ ]:
# SAM 2 video predictor over the orbit (zero-indexed jpgs, as in phase 5)
import shutil
os.makedirs('/content/sam2_frames', exist_ok=True)
for i, p in enumerate(sorted(glob.glob('/content/orbit_frames/*.jpg'))):
    shutil.copy(p, f'/content/sam2_frames/{i:05d}.jpg')
!pip -q install --force-reinstall "huggingface_hub==0.36.2"
!pip -q install "git+https://github.com/facebookresearch/sam2.git"
from sam2.sam2_video_predictor import SAM2VideoPredictor
predictor = SAM2VideoPredictor.from_pretrained('facebook/sam2.1-hiera-large')

In [ ]:
# The clicks — the only manual step. The coordinates below are PLACEHOLDERS:
# adjust (x, y) until each X sits on its target in YOUR anchor frame,
# re-running the cell to check. Targets on garden: the central vase/plant, its
# shadow on the table, its reflection in the glossy tabletop. Add extra clicks
# to a region's list if SAM2's propagation drifts.
import matplotlib.pyplot as plt

ANCHOR = 40
CLICKS = {1: ('vase',             [(416, 210)]),
          2: ('table shadow',     [(430, 300)]),
          3: ('table reflection', [(400, 330)])}

plt.figure(figsize=(12, 7))
plt.imshow(Image.open(f'/content/sam2_frames/{ANCHOR:05d}.jpg'))
for oid, (name, pts) in CLICKS.items():
    for x, y in pts:
        plt.scatter([x], [y], s=120, marker='x')
        plt.annotate(name, (x, y), color='yellow', xytext=(x + 12, y - 10))
plt.axis('off'); plt.show()

In [ ]:
# Propagate, union, dilate -> mask video; verify coverage start / middle / end
from PIL import ImageFilter

state = predictor.init_state(video_path='/content/sam2_frames')
for oid, (name, pts) in CLICKS.items():
    predictor.add_new_points_or_box(
        inference_state=state, frame_idx=ANCHOR, obj_id=oid,
        points=np.array(pts, np.float32), labels=np.ones(len(pts), np.int32))

N = 81
union = np.zeros((N, H, W), bool)
with torch.inference_mode():
    for reverse in (False, True):
        for fidx, obj_ids, logits in predictor.propagate_in_video(state, reverse=reverse):
            for i in range(len(obj_ids)):
                union[fidx] |= (logits[i, 0] > 0).cpu().numpy()
print('mean coverage:', round(float(union.mean()), 3))

masks = []
for i in range(N):
    m = Image.fromarray((union[i] * 255).astype('uint8')).filter(ImageFilter.MaxFilter(15))
    masks.append(m.point(lambda v: 255 if v > 127 else 0))
export_to_video([m.convert('RGB') for m in masks], '/content/mask_sam2.mp4', fps=16)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, idx in zip(ax, (0, 40, 80)):
    base = np.asarray(Image.open(f'/content/sam2_frames/{idx:05d}.jpg')).copy()
    s = np.array(masks[idx]) > 127
    base[s] = (0.5 * base[s] + [127, 0, 0]).astype('uint8')
    a.imshow(base); a.set_title(f'frame {idx + 1}'); a.axis('off')
plt.show()

del predictor, state; torch.cuda.empty_cache()

In [ ]:
# ROSE setup (same pins and weight relocation as phase 4b), then inference
%cd /content
!pip -q install "transformers==4.46.2" "tokenizers>=0.20,<0.21"
!pip -q install --force-reinstall "huggingface_hub==0.36.2"
import shutil
from huggingface_hub import snapshot_download
if not os.path.isdir('/content/ROSE'):
    !git clone https://github.com/Kunbyte-AI/ROSE.git
if not os.path.isdir('/content/ROSE/models/Wan2.1-Fun-1.3B-InP'):
    snapshot_download('alibaba-pai/Wan2.1-Fun-1.3B-InP',
                      local_dir='/content/ROSE/models/Wan2.1-Fun-1.3B-InP')
if not os.path.exists('/content/ROSE/weights/transformer/config.json'):
    snapshot_download('Kunbyte/ROSE', local_dir='/content/ROSE/weights')
    os.makedirs('/content/ROSE/weights/transformer', exist_ok=True)
    for f in ('config.json', 'diffusion_pytorch_model.safetensors'):
        if os.path.exists(f'/content/ROSE/weights/{f}'):
            shutil.move(f'/content/ROSE/weights/{f}',
                        f'/content/ROSE/weights/transformer/{f}')
%cd /content/ROSE
!python inference.py --validation_videos /content/orbit.mp4 \
  --validation_masks /content/mask_sam2.mp4 \
  --validation_prompts "remove the object" \
  --output_dir /content/rose_out_garden --video_length 81 --sample_size 480 832

In [ ]:
# Decode the edit, refit a fresh 3DGS on it (same 81 cameras), visual check
import imageio.v3 as iio
vid = sorted(glob.glob('/content/rose_out_garden/**/*.mp4', recursive=True))[0]
ds_dir = '/content/data/edited_garden'
os.makedirs(f'{ds_dir}/rgb', exist_ok=True)
for i, fr in enumerate(iio.imread(vid, plugin='pyav'), start=1):
    Image.fromarray(fr).resize((W, H)).save(f'{ds_dir}/rgb/{i:04d}.png')

meta = json.load(open('/content/orbit_transforms.json'))
shutil.copy('/content/orbit_transforms.json', f'{ds_dir}/transforms.json')
fr = meta['frames']; test_idx = set(range(0, 81, 8))
for name, part in {'train': [f for i, f in enumerate(fr) if i not in test_idx],
                   'test':  [f for i, f in enumerate(fr) if i in test_idx],
                   'val':   [f for i, f in enumerate(fr) if i in test_idx]}.items():
    json.dump({'camera_angle_x': meta['camera_angle_x'], 'frames': part},
              open(f'{ds_dir}/transforms_{name}.json', 'w'))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open('/content/orbit_frames/0041.jpg')); ax[0].set_title('render')
ax[1].imshow(Image.open(f'{ds_dir}/rgb/0041.png')); ax[1].set_title('ROSE edit')
for a in ax: a.axis('off')
plt.show()

!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-train splatfacto \
  --data /content/data/edited_garden --output-dir /content/out_garden_edited \
  --max-num-iterations 15000 --viewer.quit-on-train-completion True \
  --vis tensorboard blender-data

In [ ]:
# Render the refit on the same cameras -> before/after side-by-side video
CFG2 = sorted(glob.glob('/content/out_garden_edited/**/config.yml', recursive=True),
              key=os.path.getmtime)[-1]
_, pipe2, _, _ = eval_setup(Path(CFG2), test_mode='inference')
os.makedirs('/content/after_frames', exist_ok=True)
for i, f in enumerate(meta['frames'], start=1):
    c2w = torch.tensor(f['transform_matrix'], dtype=torch.float32)[:3]
    cam = Cameras(camera_to_worlds=c2w[None], fx=fx, fy=fy, cx=W / 2, cy=H / 2,
                  width=W, height=H,
                  camera_type=CameraType.PERSPECTIVE).to(pipe2.device)
    with torch.no_grad():
        rgb = pipe2.model.get_outputs_for_camera(cam)['rgb'].cpu().numpy()
    Image.fromarray((rgb * 255).clip(0, 255).astype('uint8')).save(
        f'/content/after_frames/{i:04d}.png')

before = [np.asarray(Image.open(p)) for p in sorted(glob.glob('/content/orbit_frames/*.jpg'))]
after  = [np.asarray(Image.open(p)) for p in sorted(glob.glob('/content/after_frames/*.png'))]
side = [Image.fromarray(np.hstack([b, a])) for b, a in zip(before, after)]
export_to_video(side, '/content/garden_before_after.mp4', fps=16)
side[40]

In [ ]:
# Persist everything to Drive
OUT = f'{DRIVE}/checkpoints/phase6_realdemo'
os.makedirs(OUT, exist_ok=True)
for d in ('/content/orbit_frames', '/content/after_frames',
          '/content/data/edited_garden/rgb'):
    shutil.copytree(d, f'{OUT}/{os.path.basename(os.path.dirname(d)) if d.endswith("rgb") else os.path.basename(d)}',
                    dirs_exist_ok=True)
for f in ('/content/orbit.mp4', '/content/mask_sam2.mp4',
          '/content/garden_before_after.mp4', '/content/orbit_transforms.json'):
    shutil.copy(f, OUT)
for src in ('/content/out_garden', '/content/out_garden_edited'):
    shutil.copytree(src, f'{OUT}/{os.path.basename(src)}', dirs_exist_ok=True)
print('saved to', OUT)